# Stage 3 — Fine-tune InternVL2-8B + LoRA trên Google Colab

**Yêu cầu:** Runtime → Change runtime type → **T4 GPU** (hoặc A100 nếu có Colab Pro)

**Workflow:**
1. Mount Google Drive
2. Upload project + data lên Drive
3. Cài thư viện
4. Fine-tune InternVL2 + LoRA
5. Download weights về máy

## Bước 1 — Kiểm tra GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU!')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
!nvidia-smi

## Bước 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra Drive đã mount chưa
import os
print('Drive mounted:', os.path.exists('/content/drive/MyDrive'))

## Bước 3 — Upload project lên Drive

**Trên máy tính của bạn**, chạy lệnh này để zip project:
```powershell
# Chạy trong PowerShell tại D:\defect_detection
Compress-Archive -Path D:\defect_detection -DestinationPath D:\defect_detection_colab.zip
```
Sau đó upload file `defect_detection_colab.zip` lên **Google Drive/MyDrive/**

**Quan trọng:** Upload thêm thư mục `data/vlm_ann/` chứa ảnh + JSON annotations vào Drive.

In [ ]:
import os, zipfile

ZIP_PATH    = '/content/drive/MyDrive/defect_detection_colab.zip'
EXTRACT_DIR = '/content/defect_detection'

if not os.path.exists(EXTRACT_DIR):
    print('Đang giải nén...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Xong!')
else:
    print('Đã giải nén rồi, bỏ qua.')

os.chdir(EXTRACT_DIR)
print('Working dir:', os.getcwd())
!ls

## Bước 4 — Cài thư viện

In [ ]:
%%capture
!pip install transformers>=4.40 peft>=0.10 accelerate bitsandbytes
!pip install timm einops sentencepiece rich scikit-learn
!pip install flash-attn --no-build-isolation  # Tăng tốc attention trên GPU
print('Cài xong!')

In [ ]:
# Kiểm tra import
import torch, transformers, peft
print('torch      :', torch.__version__)
print('transformers:', transformers.__version__)
print('peft       :', peft.__version__)
print('CUDA:', torch.cuda.is_available())

## Bước 5 — Cấu hình cho Colab (override config)

In [ ]:
# Override configs để phù hợp với T4 GPU (16GB VRAM)
import sys
sys.path.insert(0, '/content/defect_detection')

# Patch stage3_config trước khi import
import configs.stage3_config as cfg3

cfg3.VLM_MODEL_NAME   = 'OpenGVLab/InternVL2-8B'
cfg3.LOAD_IN_8BIT     = True   # T4 16GB: dùng 8-bit để tiết kiệm VRAM
cfg3.LOAD_IN_4BIT     = False
cfg3.LORA_RANK        = 8      # Giảm từ 16 → 8 để tiết kiệm VRAM
cfg3.LORA_ALPHA       = 16
cfg3.LORA_DROPOUT     = 0.05
cfg3.TRAIN_BATCH_SIZE = 1      # T4 với 8-bit: batch=1
cfg3.GRAD_ACCUMULATION= 16     # Effective batch = 16
cfg3.NUM_EPOCHS       = 5
cfg3.LEARNING_RATE    = 2e-5

# Đường dẫn output trỏ vào Drive để tự động lưu
DRIVE_OUTPUT = '/content/drive/MyDrive/defect_detection_weights'
import os; os.makedirs(DRIVE_OUTPUT, exist_ok=True)
cfg3.LORA_WEIGHTS_PATH = f'{DRIVE_OUTPUT}/stage3_lora'

print('Config Colab:')
print(f'  Model      : {cfg3.VLM_MODEL_NAME}')
print(f'  8-bit      : {cfg3.LOAD_IN_8BIT}')
print(f'  LoRA rank  : {cfg3.LORA_RANK}')
print(f'  Batch size : {cfg3.TRAIN_BATCH_SIZE} (eff. {cfg3.TRAIN_BATCH_SIZE * cfg3.GRAD_ACCUMULATION})')
print(f'  Epochs     : {cfg3.NUM_EPOCHS}')
print(f'  Output     : {cfg3.LORA_WEIGHTS_PATH}')

## Bước 6 — Kiểm tra dataset

In [ ]:
from stage3_vlm.dataset_vlm import VLMDataset

ds = VLMDataset(augment=False)
print(f'Số sample: {len(ds)}')

if len(ds) > 0:
    item = ds[0]
    print(f'Image shape  : {item["image"].shape}')
    print(f'Caption      : {item["target"][:100]}')
    print(f'Defect type  : {item["defect_type"]}')
    print(f'Severity     : {item["severity"]}')
    dist = ds.class_distribution()
    print(f'Phân bố lỗi : {dist}')
else:
    print('CẢNH BÁO: Không tìm thấy sample!')
    print('Hãy chắc chắn đã upload thư mục data/vlm_ann/ lên Drive')

## Bước 7 — Load model InternVL2-8B + inject LoRA

In [ ]:
import torch
from stage3_vlm.model import InternVL2Wrapper

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# Load model (lần đầu sẽ download ~16GB từ HuggingFace)
model = InternVL2Wrapper(device=DEVICE, use_lora=True)

# Đếm params
trainable, total = model._count_params()
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

# Kiểm tra VRAM
print(f'VRAM đã dùng: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print(f'VRAM còn lại: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f} GB')

## Bước 8 — BẮT ĐẦU TRAIN

In [ ]:
import argparse
from stage3_vlm.lora_train import train

# Tạo args giả (thay cho argparse CLI)
args = argparse.Namespace(
    category='tsfabric_T1',
    eval=True,          # Chạy eval sau train
)

print('Bắt đầu fine-tune InternVL2 LoRA...')
train(args)
print('TRAIN XONG!')

## Bước 9 — Kiểm tra weights đã lưu vào Drive

In [ ]:
import os
lora_dir = cfg3.LORA_WEIGHTS_PATH

if os.path.exists(lora_dir):
    files = os.listdir(lora_dir)
    total_size = sum(os.path.getsize(f'{lora_dir}/{f}') for f in files) / 1e6
    print(f'LoRA weights đã lưu tại: {lora_dir}')
    print(f'Files: {files}')
    print(f'Tổng dung lượng: {total_size:.1f} MB')
else:
    print('Chưa thấy weights, kiểm tra lại!')

## Bước 10 — Download weights về máy

Weights đã tự động lưu vào Google Drive tại:
```
Google Drive/MyDrive/defect_detection_weights/stage3_lora/
```

**Trên máy tính của bạn:**
1. Mở Google Drive trên trình duyệt
2. Tải thư mục `defect_detection_weights/stage3_lora/` về
3. Đặt vào: `D:\defect_detection\outputs\checkpoints\stage3_tsfabric_T1_lora\`

Hoặc dùng lệnh dưới để zip và download:

In [ ]:
# Zip weights để download dễ hơn
import zipfile, os

zip_out = '/content/drive/MyDrive/stage3_lora_weights.zip'
lora_dir = cfg3.LORA_WEIGHTS_PATH

with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(lora_dir):
        zf.write(f'{lora_dir}/{f}', f)

size_mb = os.path.getsize(zip_out) / 1e6
print(f'Đã zip → {zip_out}')
print(f'Kích thước: {size_mb:.1f} MB')
print('Download file này từ Google Drive về máy tính!')

## (Tùy chọn) — Test inference sau khi train

In [ ]:
from stage3_vlm.inference import VLMInference
import numpy as np
from PIL import Image

# Load model với LoRA weights vừa train
vlm = VLMInference(device=DEVICE, lora_weights=cfg3.LORA_WEIGHTS_PATH)

# Test với 1 ảnh từ dataset
if len(ds) > 0:
    test_img_path, test_ann = ds.samples[0]
    test_image = np.array(Image.open(test_img_path).convert('RGB'))

    result = vlm.analyze(test_image, anomaly_score=0.85)

    print('=== KẾT QUẢ INFERENCE ===')
    print(f'Caption    : {result["caption"]}')
    print(f'Defect type: {result["defect_type"]}')
    print(f'Severity   : {result["severity"]}')
    print(f'Pass/Fail  : {result["pass_fail"]}')
    print(f'Confidence : {result["confidence"]}')